# Tratamento e Validação da CAPAG — Ano-Base 2025
### Posição setembro/2026

**Objetivo:** tratar e validar a base CAPAG utilizada no projeto
Observatório do Clima, preparando uma tabela municipal para integração
com SINISA e Atlas Digital de Desastres.

A base utilizada é a posição de setembro/2026, referente ao ano-base 2025.

## Etapas

1. Carregamento da base oficial utilizada pelo projeto.
2. Validação da versão da CAPAG.
3. Seleção das colunas necessárias.
4. Verificação de nulos e duplicidades.
5. Validação do código IBGE.
6. Padronização dos campos.
7. Criação dos critérios fiscais.
8. Validação final.
9. Exportação do CSV tratado.

## Regra de negócio

A CAPAG é um dos critérios utilizados no funil de priorização.

Além da flag histórica de municípios com CAPAG C ou D, será mantida uma
flag referente à regra fiscal aprovada pela célula em 07/10:

- C ou D;
- n.d. ou n.e.

Essas categorias formam o grupo utilizado pela regra fiscal de
priorização do projeto.

A integração com SINISA e Atlas será realizada posteriormente pelo
código IBGE de 7 dígitos.


## 1. Importações e configurações



In [57]:
import pandas as pd
import warnings
from pathlib import Path

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="openpyxl"
)

# ---------------------------------------------------------
# CONFIGURAÇÕES DA FONTE
# ---------------------------------------------------------

# CAPAG posição setembro/2026
ARQUIVO_ID = "1ctfcr6QOz5XrQwPVrjl5h_Px52J-aj2-"

URL_EXPORT = (
    f"https://docs.google.com/spreadsheets/d/"
    f"{ARQUIVO_ID}/export?format=xlsx"
)

ABA_CAPAG = "CAPAG Ano Base 2025"

# Contagens de controle da posição setembro/2026
CONTAGENS_ESPERADAS = {
    "A+": 418,
    "A": 1096,
    "B+": 200,
    "B": 963,
    "C": 2274,
    "D": 20,
    "n.d.": 426,
    "n.e.": 170
}

TOTAL_ESPERADO = sum(CONTAGENS_ESPERADAS.values())

print("Total esperado de municípios:", TOTAL_ESPERADO)

Total esperado de municípios: 5567



# 2. Carregamento da base CAPAG

Leitura da posição de setembro/2026, ano-base 2025.
O código IBGE é carregado como texto porque é um identificador.

In [58]:
df = pd.read_excel(
    URL_EXPORT,
    sheet_name=ABA_CAPAG,
    header=3,
    dtype={
        "Código Município Completo": "string"
    }
)

print("Dimensão da base:", df.shape)

df.head()

Dimensão da base: (5567, 70)


,Código Município Completo,Nome_Município,UF,2025 - Dívida Consolidada,2025 - Receita Corrente Líquida,Indicador 1,Nota 1,2023 - Despesas Empenhadas,2023 - Receitas Correntes - Receitas Brutas Realizadas,2023 - Receitas Correntes - Deduções FUNDEB,...,DCB zerada ou negativa,Publicou RGF,OF negativa,Indicador 3,Insuficiência de caixa,Indicador 3 Antigo,Nota 3,CAPAG antes do ranking,Ranking da CCONF,CAPAG
0,5200050,Abadia de Goiás,GO,679436.95,1.139670e+08,0.005962,A,6.712108e+07,7.558667e+07,-4314496.84,...,NaN,Sim,NaN,0.103345,0.00,0.000448,A,A,Bicf,A
1,3100104,Abadia dos Dourados,MG,4263740.15,4.620732e+07,0.092274,A,3.372663e+07,4.003463e+07,-4915869.26,...,NaN,Sim,NaN,0.024377,-5740.80,0.298032,B,B,Dicf,B
2,5200100,Abadiânia,GO,24620842.46,9.427693e+07,0.261154,A,8.001550e+07,8.317818e+07,-7424954.17,...,NaN,Sim,NaN,-0.017514,-164204.73,3.051798,C,C,Bicf,C
3,3100203,Abaeté,MG,41600768.20,1.147456e+08,0.362548,A,8.707676e+07,9.658018e+07,-9468925.16,...,NaN,Sim,NaN,0.008613,-96110.04,0.127758,B,C,Aicf,C
4,1500107,Abaetetuba,PA,180.00,6.306541e+08,0.0,A,4.193456e+08,5.250544e+08,-30067247.36,...,NaN,Sim,NaN,-0.071245,-35213597.01,NaN,C,C,Bicf,C


## 3. Validar se realmente é setembro

Esta etapa funciona como controle de versão.
As contagens devem corresponder à posição de setembro/2026.

In [59]:
# Padroniza apenas espaços da classificação
df["CAPAG"] = (
    df["CAPAG"]
    .astype("string")
    .str.strip()
)

distribuicao_inicial = (
    df["CAPAG"]
    .value_counts(dropna=False)
)

print(distribuicao_inicial)

CAPAG
C       2274
A       1096
B        963
n.d.     426
A+       418
B+       200
n.e.     170
D         20
Name: count, dtype: Int64


In [60]:
# Validação automática das contagens esperadas
for nota, quantidade_esperada in CONTAGENS_ESPERADAS.items():

    quantidade_encontrada = int(
        distribuicao_inicial.get(nota, 0)
    )

    assert quantidade_encontrada == quantidade_esperada, (
        f"ERRO na nota {nota}: "
        f"esperado {quantidade_esperada}, "
        f"encontrado {quantidade_encontrada}"
    )

assert len(df) == TOTAL_ESPERADO, (
    f"Quantidade de municípios inesperada: {len(df)}"
)

print("\n✅ Base CAPAG setembro/2026 validada.")


✅ Base CAPAG setembro/2026 validada.


## 4. Seleção das colunas



In [61]:
# Seleção das colunas necessárias
colunas_capag = [
    "Código Município Completo",
    "Nome_Município",
    "UF",
    "CAPAG antes do ranking",
    "Ranking da CCONF",
    "CAPAG"
]

# Confere se todas as colunas existem antes de selecionar
colunas_faltantes = [
    coluna
    for coluna in colunas_capag
    if coluna not in df.columns
]

assert not colunas_faltantes, (
    f"Colunas não encontradas: {colunas_faltantes}"
)

df_capag = df[colunas_capag].copy()

print("Dimensão da base selecionada:", df_capag.shape)

df_capag.head()

Dimensão da base selecionada: (5567, 6)


,Código Município Completo,Nome_Município,UF,CAPAG antes do ranking,Ranking da CCONF,CAPAG
0,5200050,Abadia de Goiás,GO,A,Bicf,A
1,3100104,Abadia dos Dourados,MG,B,Dicf,B
2,5200100,Abadiânia,GO,C,Bicf,C
3,3100203,Abaeté,MG,C,Aicf,C
4,1500107,Abaetetuba,PA,C,Bicf,C


## 5. Qualidade dos dados

Verificação de valores ausentes, linhas duplicadas e duplicidade
do código IBGE.

In [62]:
# Validação de nulos e duplicidades
print("Valores nulos por coluna:")
print(df_capag.isna().sum())

print("\nLinhas totalmente duplicadas:")
duplicadas = df_capag.duplicated().sum()
print(duplicadas)

print("\nCódigos IBGE duplicados:")
duplicados_ibge = (
    df_capag["Código Município Completo"]
    .duplicated()
    .sum()
)
print(duplicados_ibge)

# Campos indispensáveis
campos_obrigatorios = [
    "Código Município Completo",
    "Nome_Município",
    "UF",
    "CAPAG"
]

assert df_capag[campos_obrigatorios].isna().sum().sum() == 0, (
    "Existem valores nulos em campos obrigatórios."
)

assert duplicadas == 0, (
    "Existem linhas duplicadas."
)

assert duplicados_ibge == 0, (
    "Existem códigos IBGE duplicados."
)

print("\n✅ Nulos e duplicidades validados.")

Valores nulos por coluna:
Código Município Completo    0
Nome_Município               0
UF                           0
CAPAG antes do ranking       0
Ranking da CCONF             0
CAPAG                        0
dtype: int64

Linhas totalmente duplicadas:
0

Códigos IBGE duplicados:
0

✅ Nulos e duplicidades validados.


## 6. Código IBGE

O código IBGE é tratado como identificador textual e deve possuir
exatamente 7 dígitos.

In [63]:
# Validação do código IBGE
df_capag["Código Município Completo"] = (
    df_capag["Código Município Completo"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

codigo_valido = (
    df_capag["Código Município Completo"]
    .str.fullmatch(r"\d{7}", na=False)
)

print(
    "Códigos válidos com 7 dígitos:",
    codigo_valido.sum()
)

print(
    "Códigos inválidos:",
    (~codigo_valido).sum()
)

assert codigo_valido.all(), (
    "Foram encontrados códigos IBGE fora do padrão de 7 dígitos."
)

assert (
    df_capag["Código Município Completo"].nunique()
    == TOTAL_ESPERADO
), "Código IBGE não é único."

print("\n✅ Código IBGE validado.")

Códigos válidos com 7 dígitos: 5567
Códigos inválidos: 0

✅ Código IBGE validado.


## 7. Padronização dos campos


In [64]:

# Padronização dos campos
df_capag["Nome_Município"] = (
    df_capag["Nome_Município"]
    .astype("string")
    .str.strip()
)

df_capag["UF"] = (
    df_capag["UF"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_capag["CAPAG"] = (
    df_capag["CAPAG"]
    .astype("string")
    .str.strip()
)

df_capag["CAPAG antes do ranking"] = (
    df_capag["CAPAG antes do ranking"]
    .astype("string")
    .str.strip()
)

df_capag["Ranking da CCONF"] = (
    df_capag["Ranking da CCONF"]
    .astype("string")
    .str.strip()
)

print("Padronização concluída.")

Padronização concluída.


## 8. Padronizar nomes das colunas

In [65]:
df_capag_final = df_capag.rename(
    columns={
        "Código Município Completo": "cod_ibge",
        "Nome_Município": "municipio",
        "UF": "uf",
        "CAPAG antes do ranking": "capag_pre_ranking",
        "Ranking da CCONF": "ranking_cconf",
        "CAPAG": "capag"
    }
).copy()

df_capag_final.head()

,cod_ibge,municipio,uf,capag_pre_ranking,ranking_cconf,capag
0,5200050,Abadia de Goiás,GO,A,Bicf,A
1,3100104,Abadia dos Dourados,MG,B,Dicf,B
2,5200100,Abadiânia,GO,C,Bicf,C
3,3100203,Abaeté,MG,C,Aicf,C
4,1500107,Abaetetuba,PA,C,Bicf,C


## 9. Criar as regras fiscais
São preservados dois indicadores:

- `criterio_capag_cd`: regra anterior do projeto C/D.

- `criterio_fiscal_priorizacao`: inclui C, D, n.d. e n.e.

Também será criada uma flag específica para municípios sem nota.

In [66]:
# ---------------------------------------------------------
# REGRA ANTIGA / CONTROLE
# ---------------------------------------------------------

df_capag_final["criterio_capag_cd"] = (
    df_capag_final["capag"]
    .isin(["C", "D"])
)


# ---------------------------------------------------------
# MUNICÍPIOS SEM NOTA
# ---------------------------------------------------------

df_capag_final["sem_nota_capag"] = (
    df_capag_final["capag"]
    .isin(["n.d.", "n.e."])
)


# ---------------------------------------------------------
# REGRA FISCAL APROVADA PELA CÉLULA EM 07/10
# ---------------------------------------------------------

categorias_priorizadas = [
    "C",
    "D",
    "n.d.",
    "n.e."
]

df_capag_final["criterio_fiscal_priorizacao"] = (
    df_capag_final["capag"]
    .isin(categorias_priorizadas)
)


print("CAPAG C/D:")
print(
    df_capag_final["criterio_capag_cd"]
    .value_counts()
)

print("\nSem nota CAPAG:")
print(
    df_capag_final["sem_nota_capag"]
    .value_counts()
)

print("\nCritério fiscal de priorização:")
print(
    df_capag_final["criterio_fiscal_priorizacao"]
    .value_counts()
)

CAPAG C/D:
criterio_capag_cd
False    3273
True     2294
Name: count, dtype: int64

Sem nota CAPAG:
sem_nota_capag
False    4971
True      596
Name: count, dtype: int64

Critério fiscal de priorização:
criterio_fiscal_priorizacao
True     2890
False    2677
Name: count, dtype: int64


## 10. Validação final

In [67]:
print("Dimensão final:", df_capag_final.shape)

print("\nDistribuição CAPAG final:")
distribuicao_final = (
    df_capag_final["capag"]
    .value_counts(dropna=False)
)

print(distribuicao_final)


# ---------------------------------------------------------
# TESTES AUTOMÁTICOS
# ---------------------------------------------------------

assert len(df_capag_final) == 5567, (
    "Quantidade final de municípios incorreta."
)

assert df_capag_final["cod_ibge"].nunique() == 5567, (
    "Código IBGE não está único."
)

assert df_capag_final["cod_ibge"].duplicated().sum() == 0, (
    "Existem códigos IBGE duplicados."
)

assert (
    df_capag_final["cod_ibge"]
    .str.fullmatch(r"\d{7}", na=False)
    .all()
), "Existem códigos IBGE inválidos."


# Distribuição CAPAG
for nota, quantidade_esperada in CONTAGENS_ESPERADAS.items():

    quantidade_encontrada = int(
        distribuicao_final.get(nota, 0)
    )

    assert quantidade_encontrada == quantidade_esperada, (
        f"Erro na distribuição da nota {nota}: "
        f"{quantidade_encontrada}"
    )


# Critério antigo C/D
assert (
    df_capag_final["criterio_capag_cd"].sum()
    == 2294
), "Quantidade C/D inesperada."


# Municípios sem nota
assert (
    df_capag_final["sem_nota_capag"].sum()
    == 596
), "Quantidade de municípios sem nota inesperada."


# Nova regra fiscal
assert (
    df_capag_final["criterio_fiscal_priorizacao"].sum()
    == 2890
), "Quantidade do critério fiscal inesperada."


print("\n✅ TODAS AS VALIDAÇÕES FORAM APROVADAS.")

Dimensão final: (5567, 9)

Distribuição CAPAG final:
capag
C       2274
A       1096
B        963
n.d.     426
A+       418
B+       200
n.e.     170
D         20
Name: count, dtype: Int64

✅ TODAS AS VALIDAÇÕES FORAM APROVADAS.


## 11. Exportação

In [68]:
caminho_saida = Path(
    "/content/capag_nacional_tratada.csv"
)

df_capag_final.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Arquivo exportado com sucesso.")
print("Caminho:", caminho_saida)
print("Dimensão:", df_capag_final.shape)

✅ Arquivo exportado com sucesso.
Caminho: /content/capag_nacional_tratada.csv
Dimensão: (5567, 9)


## 12. Conferência do arquivo que realmente foi exportado

In [69]:
df_exportado = pd.read_csv(
    caminho_saida,
    dtype={
        "cod_ibge": "string"
    },
    encoding="utf-8-sig"
)

print("Dimensão do CSV exportado:", df_exportado.shape)

print("\nDistribuição da CAPAG no CSV:")
print(
    df_exportado["capag"]
    .value_counts(dropna=False)
)

assert len(df_exportado) == 5567

assert df_exportado["cod_ibge"].nunique() == 5567

assert df_exportado["cod_ibge"].duplicated().sum() == 0

print("\n✅ CSV exportado e relido com sucesso.")

Dimensão do CSV exportado: (5567, 9)

Distribuição da CAPAG no CSV:
capag
C       2274
A       1096
B        963
n.d.     426
A+       418
B+       200
n.e.     170
D         20
Name: count, dtype: int64

✅ CSV exportado e relido com sucesso.
